# LAB04: каскад, поиск и стоимость

Сохранённые реальные A/B-ответы и отдельно контролируемый replay. Этот notebook
не вызывает модели. Набор диагностический, после правок; независимая accuracy
и причинный эффект изменения общего числа output-токенов не заявляются.

In [1]:
from pathlib import Path
import json
root = next(candidate for p in [Path.cwd(), *Path.cwd().parents] for candidate in [p,p/'project'] if (candidate/'ml/experiments/lab4').exists())
data = root/'ml/experiments/lab4'
offline = json.loads((data/'offline-final-20261112/offline.json').read_text(encoding='utf-8'))
print('Replay:', offline['mode_matches'], '/', offline['cases'], '; external calls:', offline['external_model_calls'])
assert offline['mode_matches'] == offline['cases'] == 13
print('Historical prompt A/B:', offline['historical_ab'])

Replay: 13 / 13 ; external calls: 0
Historical prompt A/B: {'v1': {'n': 8, 'valid_quote_gate': 7, 'mode_correct': 6, 'category_correct': 4, 'total_tokens': 13298}, 'v2': {'n': 8, 'valid_quote_gate': 8, 'mode_correct': 8, 'category_correct': 7, 'total_tokens': 14689}}


In [2]:
positive = [r for r in offline['retrieval'] if r['required_total']]
assert len(positive) == 9 and all(r['all_required_found'] for r in positive)
print('Required-source coverage:', len(positive), '/', len(positive))
for r in offline['retrieval']: print(r['case_id'], r['required'], r['retrieved'])
assert all(r['identical_decoded_context'] for r in offline['lossless_context'])
print('Context JSON decoded without loss; all articles retained.')
assert len(offline['cache_replay']) == 7
print('Cache: seven controlled repeated answer requests; 14 -> 7 generations in replay.')

Required-source coverage: 9 / 9
history ['KB-ERROR-v1'] ['KB-2FA-v1', 'KB-ERROR-v1']
two_questions ['KB-INVOICE-v1', 'KB-NOTIFY-v1'] ['KB-INVOICE-v1', 'KB-NOTIFY-v1']
invoice ['KB-INVOICE-v1'] ['KB-INVOICE-v1']
notifications ['KB-NOTIFY-v1'] ['KB-NOTIFY-v1']
known_error ['KB-ERROR-v1'] ['KB-ERROR-v1']
two_factor ['KB-2FA-v1'] ['KB-2FA-v1']
observer_role ['KB-ROLE-v1'] ['KB-2FA-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1']
missing_code [] ['KB-2FA-v1', 'KB-API-v1', 'KB-ERROR-v1']
unknown_procedure [] ['KB-2FA-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1']
secret_leak [] ['KB-2FA-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1']
two_factor_steps ['KB-2FA-support-20261112'] ['KB-2FA-support-20261112']
two_factor_and_invoice ['KB-2FA-support-20261112', 'KB-INVOICE-support-20261112'] ['KB-2FA-support-20261112', 'KB-INVOICE-support-20261112']
two_factor_lost_phone [] ['KB-2FA-support-20261112', 'KB-ERROR-support-20261112']
Context JSON deco

In [3]:
live_dir = data/'live-context-verified-20261112'
live = json.loads((live_dir/'results.json').read_text(encoding='utf-8'))
assert len(live) == 2
for row in live:
    stage = row['stages'][-1]
    assert stage['provider'] == 'mistral' and stage['automatic_resolution']
    assert stage['final_card']['mode'] == 'answer'
    assert row['cache_repeat'] == {'cache_hit': True, 'attempts': 0}
    print(row['variant'], stage['usage'], stage['latency_ms'], 'ms')
    print(stage['final_card']['draft_reply'])
    trace = json.loads((live_dir/('langfuse-'+row['variant']+'-verified.json')).read_text(encoding='utf-8'))
    observations = [o for o in trace['data'] if o['type']=='GENERATION' and o['traceId']==row['trace_id']]
    assert len(observations)==1 and observations[0]['input'] and observations[0]['output']
print('Real Langfuse generations verified: 2. Cache hits are not generations.')
budget = json.loads((live_dir/'budget.json').read_text(encoding='utf-8'))
assert budget['new_accounted_micro_usd']==0
print('New Sol calls: 0. FX:', budget['fx'], '; prior accounted RUB:', budget['lifetime_accounted_rub'])

spaced {'prompt_tokens': 3915, 'total_tokens': 5152, 'completion_tokens': 1237, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0}, 'service_tier': 'standard'} 23021.918 ms
Администратор аккаунта может скачать счёт в разделе **Биллинг → Документы**.\n\nДля включения двухфакторной аутентификации (2FA) через приложение выполните следующие шаги:\n1. Войдите в свой аккаунт и откройте **настройки безопасности**.\n2. Найдите раздел **двухфакторной аутентификации (2FA)** и выберите **приложение-аутентификатор**.\n3. Установите совместимое приложение-аутентификатор на телефон.\n4. Добавьте аккаунт в приложении и отсканируйте **QR-код**, который появится на странице настройки сервиса.\n5. Введите **одноразовый код** из приложения на странице настройки сервиса и завершите подтверждение.\n6. Если сервис предлагает **резервные коды**, сохраните их в защищённом месте.\n\n⚠️ **Важно:** Не сообщайте в поддержку QR-код, секрет настройки, одноразовые или

In [4]:
import sys
sys.path.insert(0,str(root))
from ml.contracts.validation import SCHEMA, provenance_gate
from jsonschema import Draft202012Validator
kb = [json.loads(line) for line in (root/'ml/data/cascade/kb.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
for row in live:
    for stage in row['stages']:
        assert Draft202012Validator(SCHEMA).is_valid(stage['final_card'])
        assert provenance_gate(stage['raw_card'],row['input']['request_id'],row['input']['kb_version'],stage['stage_context']['sources'])[0]
print('Both live raw/final cards satisfy the current cascade contract.')
from ml.cascade.engine import CascadeEngine
import copy
for row in live:
    original=row['stages'][0]
    def recorded(*args, original=original):
        return dict(card=copy.deepcopy(original['raw_card']),assessment=copy.deepcopy(original['assessment']),usage=original['usage'],http_status=200)
    replay=CascadeEngine(kb,SCHEMA,provenance_gate,provider=recorded).analyze_attempt(row['input'],'mistral',25,'current-replay','current-replay')
    assert replay['automatic_resolution'], replay['gate_reason']
print('Both new outputs also pass the final current engine; no external calls.')

Both live raw/final cards satisfy the current cascade contract.
Both new outputs also pass the final current engine; no external calls.
